# 8. Built-in Database — `sqlite3`
**Python Libraries Lab · Standard Library**

_A real SQL database in one file, no server needed._

SQLite is a full database that lives in a single file. There is no server to install, and it ships with Python. You talk to it with SQL, the standard language for asking questions of tables, like show me total sales per city, sorted from highest.

**In this notebook:** You will build the ShopEasy orders database, add orders, then answer real business questions with SQL, like revenue per city and the top customers.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Nothing to install: this is part of Python's **standard library**, so it works everywhere Python runs.

## Step 1 · Connect and create a table
`connect` creates the file if it does not exist.

In [ ]:
import sqlite3

conn = sqlite3.connect("shopeasy.db")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("""
    CREATE TABLE orders (
        id       INTEGER PRIMARY KEY,
        customer TEXT NOT NULL,
        city     TEXT,
        amount   REAL
    )
""")
print("Table created")

## Step 2 · Insert rows safely
`?` placeholders keep you safe from SQL injection. `executemany` inserts a list in one go.

In [ ]:
rows = [
    ("Asha", "Pune", 499), ("Ravi", "Delhi", 1299), ("Meera", "Pune", 899),
    ("John", "Mumbai", 199), ("Asha", "Pune", 2999), ("Ravi", "Delhi", 499),
]
cur.executemany("INSERT INTO orders (customer, city, amount) VALUES (?, ?, ?)", rows)
conn.commit()
print(cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0], "orders saved")

## Step 3 · Ask questions with SELECT
`GROUP BY` totals per group, `ORDER BY ... DESC` sorts biggest first.

In [ ]:
query = """
    SELECT city, COUNT(*) AS orders, SUM(amount) AS revenue
    FROM orders
    GROUP BY city
    ORDER BY revenue DESC
"""
for city, n, revenue in cur.execute(query):
    print(f"{city:<7} {n} orders  Rs.{revenue:,.0f}")

## Step 4 · Filter with a parameter
The same `?` trick for WHERE clauses.

In [ ]:
min_amount = 800
big = cur.execute("SELECT customer, amount FROM orders WHERE amount >= ? ORDER BY amount DESC", (min_amount,)).fetchall()
print(big)

## Step 5 · Update, delete and rows as dicts
`row_factory = sqlite3.Row` lets you use column names.

In [ ]:
cur.execute("UPDATE orders SET amount = amount * 0.9 WHERE city = ?", ("Delhi",))
cur.execute("DELETE FROM orders WHERE amount < ?", (200,))
conn.commit()

conn.row_factory = sqlite3.Row
for r in conn.execute("SELECT * FROM orders"):
    print(dict(r))
conn.close()

## ✍️ Your turn
Reconnect to `shopeasy.db` and store the **total amount spent by Asha** (a number) in `answer`, using SQL with a `?` placeholder.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
conn = sqlite3.connect("shopeasy.db")
answer = None  # your code here

In [ ]:
def check():
    return answer == 3498, f"Expected 3498, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
conn = sqlite3.connect("shopeasy.db")
answer = conn.execute("SELECT SUM(amount) FROM orders WHERE customer = ?", ("Asha",)).fetchone()[0]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **SQL**
- **Table**
- **Connection**
- **Cursor**
- **Parameter (?)**
- **Commit**

**Next:** 9. Doing Things at Once (`asyncio · threading · multiprocessing`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/09_asyncio_threading_multiprocessing.ipynb)

## 💻 Run it on your own computer (optional)
1. Install Python 3.10+ from python.org (on Windows tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
# nothing to install: it is built into Python
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that just opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain `.py` file, run `!` commands in the terminal instead.